# Playground: char-level transformer (week-01)

Interactive scratchpad for the model in `src/main.py` / `src/mytransformer.py`.
Run `src/ablate.py` first (it trains all four ablation runs and writes `results/<run_name>/{model_weights.pt, loss_history.json, config.json}` plus `results/ablation_table.{md,json}`) — this notebook only loads and pokes at the results, it doesn't train.

The four runs, one swap at a time from a shared baseline (see `src/ablate.py`):
- **baseline** - LayerNorm + learned positional embeddings + GELU MLP
- **rmsnorm** - baseline with RMSNorm instead of LayerNorm
- **rope** - baseline with RoPE instead of learned positional embeddings
- **swiglu** - baseline with SwiGLU instead of the GELU MLP

Sections:
1. Setup - load the tokenizer and all four trained models
2. Ablation table + loss curves - all four runs compared
3. Text generation playground - prompts, temperature, side-by-side across runs
4. Per-character surprise / perplexity on your own text
5. Weight tying check
6. Learned positional embeddings - similarity structure (learned-pos runs only)
7. Attention heatmaps - what each head attends to
8. Scratch space for your own experiments

## 1. Setup

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

# Notebook lives in notebooks/, project code in src/ - adjust if you move this file.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
RESULTS_DIR = PROJECT_ROOT / "results"
DATA_DIR = PROJECT_ROOT / "data"
sys.path.insert(0, str(SRC_DIR))

from main import Transformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")

In [ ]:
# Tokenizer - must match train.py exactly (sorted unique chars of the same data file)
text = (DATA_DIR / "input.txt").read_text()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

print(f"vocab_size: {vocab_size}")
print(''.join(chars))

Load every ablation run found under `results/` (each run's `config.json` records the exact hyperparameters it was trained with, so we can reconstruct the right `Transformer` for each - no need to hardcode `d_model` etc. per run, or keep them in sync with `train.py` by hand).

In [ ]:
# Discover ablation runs: any results/<run_name>/ directory containing a config.json.
# Preferred order matches ablate.py's RUNS dict; any extra/renamed runs are appended after.
PREFERRED_ORDER = ["baseline", "rmsnorm", "rope", "swiglu"]
run_dirs = sorted(
    (d for d in RESULTS_DIR.iterdir() if d.is_dir() and (d / "config.json").exists()),
    key=lambda d: PREFERRED_ORDER.index(d.name) if d.name in PREFERRED_ORDER else len(PREFERRED_ORDER),
)
assert run_dirs, f"No ablation runs found under {RESULTS_DIR} - run src/ablate.py first."

import json

runs = {}
for run_dir in run_dirs:
    config = json.loads((run_dir / "config.json").read_text())
    # config.json stores every constructor arg Transformer needs except vocab_size handling -
    # it's saved too, so this works even if you retrain with a different tokenizer/vocab.
    model = Transformer(
        d_model=config["d_model"],
        num_layers=config["num_layers"],
        num_heads=config["num_heads"],
        vocab_size=config["vocab_size"],
        block_size=config["block_size"],
        positional_embedding_type=config["positional_embedding_type"],
        norm_type=config["norm_type"],
        mlp_type=config["mlp_type"],
    ).to(device)
    model.load_state_dict(torch.load(run_dir / "model_weights.pt", map_location=device))
    model.eval()

    loss_history = json.loads((run_dir / "loss_history.json").read_text())

    runs[run_dir.name] = {"model": model, "config": config, "loss_history": loss_history}
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Loaded '{run_dir.name}': {config['norm_type']} / {config['positional_embedding_type']} "
          f"/ {config['mlp_type']}, {n_params:,} params")

# Most of the single-model sections below (4, 5, 6, 7) inspect one run at a time - change this
# to look at a different run without re-running the loading cell above.
SELECTED_RUN = "baseline"
model = runs[SELECTED_RUN]["model"]
print(f"\nSELECTED_RUN = {SELECTED_RUN!r}")

## 2. Ablation table + loss curves

Linear scale would be misleading here: a huge first-step loss (before the weight-tying init fix, ~159 instead of the ~ln(vocab_size) a random model should give) dwarfs the y-axis and makes the rest of the curve look artificially flat/converged. Log scale plus the random-guess baseline gives an honest read across all four runs at once.

In [ ]:
# Fixed color per run, assigned in a fixed order (never re-cycled if a run is missing) so a run's
# color always means the same thing across every plot in this notebook.
RUN_COLORS = {
    "baseline": "#2a78d6",  # blue
    "rmsnorm": "#eb6834",   # orange
    "rope": "#1baf7a",      # aqua
    "swiglu": "#eda100",    # yellow
}

random_guess_loss = np.log(vocab_size)

plt.figure(figsize=(9, 5.5))
for run_name, run in runs.items():
    color = RUN_COLORS.get(run_name, "#4a3aa7")  # violet fallback for an unexpected run name
    history = run["loss_history"]
    train_steps, train_vals = zip(*history["train"])
    val_steps, val_vals = zip(*history["val"])
    plt.plot(train_steps, train_vals, color=color, alpha=0.25, linewidth=1)  # faint - train loss is noisy per-step
    plt.plot(val_steps, val_vals, color=color, linewidth=2, marker='o', markersize=4, label=run_name)

plt.axhline(random_guess_loss, color='gray', linestyle='--', linewidth=1,
            label=f'ln(vocab_size)={random_guess_loss:.2f} (random guess)')
plt.yscale('log')
plt.xlabel('training step')
plt.ylabel('cross-entropy loss (log scale)')
plt.title('Training (faint) and validation (solid, marked) loss - all ablation runs')
plt.legend()
plt.tight_layout()
plt.show()

# --- Ablation table (written by ablate.py; re-derived here so it's always in sync with what's loaded) ---
table_path = RESULTS_DIR / "ablation_table.json"
if table_path.exists():
    table = json.loads(table_path.read_text())
    print(f"{'run':<10} {'final val loss':>15} {'best epoch':>11} {'wall-clock (s)':>15}")
    for row in table:
        print(f"{row['run_name']:<10} {row['final_val_loss']:>15.4f} {row['best_epoch'] + 1:>11} "
              f"{row['wall_clock_seconds']:>15.1f}")
else:
    print(f"No {table_path} found - it's written at the end of ablate.py, after all 4 runs finish.")

## 3. Text generation playground

In [ ]:
def generate_text(model, prompt="", max_new_tokens=300, temperature=1.0):
    """Sample from the given model. Empty prompt seeds with token id 0, same as train.py."""
    if prompt:
        unknown = set(prompt) - set(stoi)
        if unknown:
            raise ValueError(f"prompt contains characters not in the training vocab: {unknown}")
        ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    else:
        ids = torch.zeros((1, 1), dtype=torch.long, device=device)
    out_ids = model.generate(ids, max_new_tokens=max_new_tokens, temperature=temperature)[0].tolist()
    return decode(out_ids)

In [ ]:
# Side-by-side: same prompt/temperature, every run - qualitative comparison of what each swap
# actually did to the generated text, not just to the loss number.
for run_name, run in runs.items():
    print(f"--- {run_name} ---")
    print(generate_text(run["model"], prompt="ROMEO:", max_new_tokens=300, temperature=0.8))
    print()

Temperature sweep — lower temperature sharpens the distribution (more repetitive/greedy), higher flattens it (more random, less coherent). This one uses `SELECTED_RUN` only, to keep the output manageable - change `SELECTED_RUN` in the setup cell to sweep a different run.

In [ ]:
for temperature in [0.3, 0.6, 0.8, 1.0, 1.3]:
    print(f"--- {SELECTED_RUN} | temperature={temperature} ---")
    print(generate_text(model, prompt="ROMEO:", max_new_tokens=150, temperature=temperature))
    print()

## 4. Per-character surprise / perplexity

Feed the model a snippet of real (or made-up) text and look at the per-character loss - where is the model confident vs. surprised? Useful for spotting whether it's learned things like "a colon usually follows a character name" or "quotes/parens close." Uses `SELECTED_RUN`.

In [ ]:
def per_char_loss(model, text_snippet):
    ids = torch.tensor([encode(text_snippet)], dtype=torch.long, device=device)
    inputs, targets = ids[:, :-1], ids[:, 1:]
    with torch.no_grad():
        logits = model(inputs)
    losses = torch.nn.functional.cross_entropy(
        logits.reshape(-1, vocab_size), targets.reshape(-1), reduction='none',
    )
    return losses.cpu().numpy()


snippet = "To be, or not to be, that is the question:"
losses = per_char_loss(model, snippet)

for ch, loss in zip(snippet[1:], losses):
    bar = '#' * int(loss * 4)
    print(f"{ch!r:>6}  loss={loss:5.2f}  {bar}")

print(f"\n[{SELECTED_RUN}] mean loss: {losses.mean():.3f}   perplexity: {np.exp(losses.mean()):.2f}")

## 5. Weight tying check

`main.py` ties `head.weight` to `token_embedding.weight` and sets `head.bias=False` (see the training-fix discussion) - confirm that actually held after loading, for every run (it's a structural property of the model class, not something an ablation swap should touch).

In [ ]:
for run_name, run in runs.items():
    m = run["model"]
    tied = m.head.weight is m.token_embedding.weight
    print(f"{run_name:<10} tied={tied}  head.bias={m.head.bias}  "
          f"token_embedding.weight.std={m.token_embedding.weight.std().item():.4f}")

## 6. Learned positional embeddings

If position information was actually learned (rather than the model ignoring it), nearby positions should look more similar to each other than distant ones - i.e. the heatmap below should show a bright band along the diagonal, not uniform noise.

Only meaningful for runs with `positional_embedding_type == 'learned'` (baseline, rmsnorm, swiglu under the one-swap-at-a-time design) - the `rope` run has no such table (position is injected inside attention instead, see `mytransformer.apply_rope`), so it's skipped below.

In [ ]:
learned_runs = {name: run for name, run in runs.items() if run["config"]["positional_embedding_type"] == "learned"}
skipped = set(runs) - set(learned_runs)
if skipped:
    print(f"Skipping (no learned position table): {sorted(skipped)}")

fig, axes = plt.subplots(1, len(learned_runs), figsize=(5 * len(learned_runs), 4.5), squeeze=False)
for ax, (run_name, run) in zip(axes[0], learned_runs.items()):
    pos_emb = run["model"].position_embedding.weight.detach().cpu()  # (block_size, d_model)
    pos_emb_norm = pos_emb / pos_emb.norm(dim=-1, keepdim=True)
    cos_sim = (pos_emb_norm @ pos_emb_norm.T).numpy()
    im = ax.imshow(cos_sim, cmap='viridis')
    ax.set_title(run_name)
    ax.set_xlabel('position')
    ax.set_ylabel('position')
    fig.colorbar(im, ax=ax, label='cosine similarity', fraction=0.046)
fig.suptitle('Positional embedding similarity')
plt.tight_layout()
plt.show()

## 7. Attention heatmaps

`mytransformer.py` caches the last attention weights on each block (`block.last_attention_weights`) purely for inspection like this - run a forward pass, then look at what each head attended to. Uses `SELECTED_RUN` - change it in the setup cell to compare e.g. `rope`'s attention pattern against `baseline`'s.

In [ ]:
block_size = runs[SELECTED_RUN]["config"]["block_size"]
sample = "ROMEO: But soft, what light through yonder window breaks?"
sample = sample[:block_size]
ids = torch.tensor([encode(sample)], dtype=torch.long, device=device)
with torch.no_grad():
    _ = model(ids)

LAYER = 0  # try other layers - model.blocks has num_layers of these
attn = model.blocks[LAYER].last_attention_weights[0].cpu().numpy()  # (num_heads, seq, seq)
num_heads = attn.shape[0]
chars_list = list(sample)

fig, axes = plt.subplots(1, num_heads, figsize=(3.2 * num_heads, 3.5))
for h, ax in enumerate(axes):
    ax.imshow(attn[h], cmap='viridis')
    ax.set_title(f'head {h}')
    ax.set_xticks(range(len(chars_list)))
    ax.set_xticklabels(chars_list, rotation=90, fontsize=6)
    ax.set_yticks(range(len(chars_list)))
    ax.set_yticklabels(chars_list, fontsize=6)
fig.suptitle(f'[{SELECTED_RUN}] Layer {LAYER} attention weights (rows=query pos, cols=key pos)')
plt.tight_layout()
plt.show()

## 8. Scratch space

Your own experiments below.